In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import os, time, csv, shutil
import pandas as pd
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.metrics import f1_score
import random
import time
from torch.utils.data import TensorDataset
from torchvision.transforms import v2
from torch.utils.data.dataloader import default_collate
import torch.nn.functional as F

In [2]:
cinic_mean_RGB = [0.47889522, 0.47227842, 0.43047404]
cinic_std_RGB  = [0.24205776, 0.23828046, 0.25874835]

SEEDS      = [42, 123, 2024, 7, 999]
NUM_EPOCHS = 25
DATA_PATH  = '/kaggle/input/datasets/mengcius/cinic10/'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

Using device: cpu


In [3]:
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
])

image_aug_pipelines = {
    'flip': transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'blur': transforms.Compose([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'jitter': transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutout': transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.25, 0.25), ratio=(1, 1), value=0),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha1': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha4': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
}

batch_aug_pipelines = {
    'cutmix_alpha1': transforms.v2.CutMix(num_classes=10, alpha=1.0),
    'cutmix_alpha4': transforms.v2.CutMix(num_classes=10, alpha=4.0),
}

def preload_to_ram(dataset):
    loader = DataLoader(dataset, batch_size=512, num_workers=4, pin_memory=False)
    all_images, all_labels = [], []
    for images, labels in loader:
        all_images.append(images)
        all_labels.append(labels)
    return TensorDataset(torch.cat(all_images), torch.cat(all_labels))

def get_cutmix_collate_fn(aug_type, p=0.5):
    def collate_fn(batch):
        cutmix = batch_aug_pipelines[aug_type]
        images, labels = default_collate(batch)
        if torch.rand(()) < p:
            images, labels = cutmix(images, labels)
        return images, labels
    return collate_fn

In [4]:
_loader_cache = {}

def get_loaders(batch_size: int, path: str = DATA_PATH, aug_type: str = None):
    train_base = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=base_transform)
    valid_ds   = datasets.ImageFolder(root=os.path.join(path, 'valid'), transform=base_transform)
    test_ds    = datasets.ImageFolder(root=os.path.join(path, 'test'),  transform=base_transform)

    key = (batch_size, aug_type)
    if key in _loader_cache:
        return _loader_cache[key]

    valid_ds = preload_to_ram(valid_ds)
    test_ds  = preload_to_ram(test_ds)

    if aug_type is None:
        train_base = preload_to_ram(train_base)
        train_ds = train_base
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=4, pin_memory=True, persistent_workers=True)
    else:
        if aug_type in ('cutmix_alpha1', 'cutmix_alpha4'):
            chosen_transform = image_aug_pipelines[aug_type]
            train_aug = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=chosen_transform)
            train_ds  = ConcatDataset([train_base, train_aug])
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                      num_workers=4, pin_memory=True, persistent_workers=True,
                                      collate_fn=get_cutmix_collate_fn(aug_type))
        else:
            if aug_type == 'random':
                standard_transforms = {k: v for k, v in image_aug_pipelines.items() 
                                       if k not in ('cutmix_alpha1', 'cutmix_alpha4')}
                chosen_transform = transforms.RandomChoice(list(standard_transforms.values()))
            else:
                chosen_transform = image_aug_pipelines[aug_type]
            train_aug = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=chosen_transform)
            train_ds  = ConcatDataset([train_base, train_aug])
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                      num_workers=4, pin_memory=True, persistent_workers=True)

    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)

    _loader_cache[key] = (train_loader, valid_loader, test_loader)
    return _loader_cache[key]

In [5]:
def get_resnet18_for_cinic10(num_classes=10, use_pretrained=False, dropout=0.2, freeze_pretrained=False):
    weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
    model = models.resnet18(weights=weights)
    if use_pretrained and freeze_pretrained:
        for param in model.parameters():
            param.requires_grad = False
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    num_ftrs = model.fc.in_features
    if dropout > 0:
        model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(num_ftrs, num_classes))
    else:
        model.fc = nn.Linear(num_ftrs, num_classes)
    return model

def get_mobilenetv2(num_classes=10, dropout_rate=0.0):
    m = models.mobilenet_v2()
    old_conv = m.features[0][0]
    m.features[0][0] = nn.Conv2d(
        in_channels=old_conv.in_channels,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=1,
        padding=old_conv.padding,
        dilation=old_conv.dilation,
        groups=old_conv.groups,
        bias=(old_conv.bias is not None)
    )
    m.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(m.classifier[1].in_features, num_classes)
    )
    return m

In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, use_residual: bool = False):
        super().__init__()
        self.use_residual = use_residual

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        ) if use_residual and in_ch != out_ch else nn.Identity()

        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        out = self.block(x)
        if self.use_residual:
            out = out + self.skip(x)
        out = self.relu(out)
        out = self.pool(out)
        return out

In [7]:
class CustomCNN(nn.Module):
    """
    Architecture:
        Block 1: 3   → 32  (no residual)   32×32 → 16×16
        Block 2: 32  → 64  (no residual)   16×16 → 8×8
        Block 3: 64  → 128 (residual)      8×8   → 4×4
        Block 4: 128 → 256 (residual)      4×4   → 2×2
        GAP → Dropout → FC(256, 10)

    """
    def __init__(self, num_classes: int = 10, dropout: float = 0.0):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(3,   32,  use_residual=False),
            ConvBlock(32,  64,  use_residual=False),
            ConvBlock(64,  128, use_residual=True),
            ConvBlock(128, 256, use_residual=True),
        )

        self.gap        = nn.AdaptiveAvgPool2d(1)
        self.dropout    = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(256, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [8]:
def load_model(model, path, key=None):
    state = torch.load(path, map_location=DEVICE)
    if key:
        model.load_state_dict(state[key])
    else:
        model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model

def get_model_paths(models_dir: str):
    all_files = sorted([f for f in os.listdir(models_dir) if f.endswith('.pth')])
    resnet_paths    = [os.path.join(models_dir, f) for f in all_files if f.startswith('resnet')]
    mobilenet_paths = [os.path.join(models_dir, f) for f in all_files if f.startswith('mobilenet')]
    custom_paths    = [os.path.join(models_dir, f) for f in all_files if f.startswith('custom')]
    return resnet_paths, mobilenet_paths, custom_paths

def load_all_models(models_dir: str):
    resnet_paths, mobilenet_paths, custom_paths = get_model_paths(models_dir)
    
    all_models = []
    
    # ResNet18
    for path in resnet_paths:
        model = get_resnet18_for_cinic10(num_classes=10, use_pretrained=False, dropout=0.2, freeze_pretrained=False)
        model = load_model(model, path, key=None)
        all_models.append(model)
        print(f"Loaded ResNet18: {path}")
    
    # MobileNetV2
    for path in mobilenet_paths:
        model = get_mobilenetv2(num_classes=10, dropout_rate=0.0)
        model = load_model(model, path, key="model")
        all_models.append(model)
        print(f"Loaded MobileNetV2: {path}")
    
    # CustomCNN
    for path in custom_paths:
        model = CustomCNN(num_classes=10, dropout=0.3)
        model = load_model(model, path, key=None)
        all_models.append(model)
        print(f"Loaded CustomCNN: {path}")
    
    return all_models

@torch.no_grad()
def ensemble_predict(models, loader, device):
    """
    Soft voting
    """
    all_preds  = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        
        probs_sum = torch.zeros(images.size(0), 10).to(device)
        for model in models:
            logits = model(images)
            probs  = F.softmax(logits, dim=1)
            probs_sum += probs
    
        avg_probs = probs_sum / len(models)
        preds = avg_probs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return f1, all_preds, all_labels

@torch.no_grad()
def ensemble_predict_hard(models, loader, device):
    """
    Hard voting
    """
    all_preds  = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        votes = torch.zeros(len(models), images.size(0), dtype=torch.long).to(device)
        for i, model in enumerate(models):
            logits = model(images)
            votes[i] = logits.argmax(dim=1)

        preds, _ = torch.mode(votes, dim=0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return f1, all_preds, all_labels

In [9]:
MODELS_DIR = '/kaggle/input/datasets/mikoajrowicki/best-models'

print("Loading all models...")
all_models = load_all_models(MODELS_DIR)
print(f"\nTotal models in ensemble: {len(all_models)}")

_, _, test_loader = get_loaders(batch_size=64)

print("\nRunning ensemble on test set...")
test_f1, preds, labels = ensemble_predict(all_models, test_loader, DEVICE)
print(f"\n Ensemble Test F1: {test_f1:.4f}")


Loading all models...
Loaded ResNet18: /kaggle/input/datasets/mikoajrowicki/best-models/resnet1.pth
Loaded ResNet18: /kaggle/input/datasets/mikoajrowicki/best-models/resnet2.pth
Loaded ResNet18: /kaggle/input/datasets/mikoajrowicki/best-models/resnet3.pth
Loaded MobileNetV2: /kaggle/input/datasets/mikoajrowicki/best-models/mobilenet1.pth
Loaded MobileNetV2: /kaggle/input/datasets/mikoajrowicki/best-models/mobilenet2.pth
Loaded MobileNetV2: /kaggle/input/datasets/mikoajrowicki/best-models/mobilenet3.pth
Loaded MobileNetV2: /kaggle/input/datasets/mikoajrowicki/best-models/mobilenet4.pth
Loaded CustomCNN: /kaggle/input/datasets/mikoajrowicki/best-models/custom1.pth
Loaded CustomCNN: /kaggle/input/datasets/mikoajrowicki/best-models/custom2.pth
Loaded CustomCNN: /kaggle/input/datasets/mikoajrowicki/best-models/custom3.pth
Loaded CustomCNN: /kaggle/input/datasets/mikoajrowicki/best-models/custom4.pth
Loaded CustomCNN: /kaggle/input/datasets/mikoajrowicki/best-models/custom5.pth

Total models

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



 Ensemble Test F1: 0.8412


In [10]:
print("\nRunning hard voting ensemble on test set...")
test_f1_hard, preds_hard, _ = ensemble_predict_hard(all_models, test_loader, DEVICE)
print(f" Hard Voting Test F1: {test_f1_hard:.4f}")

ensemble_results = pd.DataFrame([{
    'n_models':    len(all_models),
    'n_resnet':    len([f for f in os.listdir(MODELS_DIR) if f.startswith('resnet') and f.endswith('.pth')]),
    'n_mobilenet': len([f for f in os.listdir(MODELS_DIR) if f.startswith('mobilenet') and f.endswith('.pth')]),
    'n_custom':    len([f for f in os.listdir(MODELS_DIR) if f.startswith('custom') and f.endswith('.pth')]),
    'test_f1_soft': test_f1,
    'test_f1_hard': test_f1_hard,
}])

ensemble_results.to_csv('ensemble_results.csv', index=False)
print(" Results saved to ensemble_results.csv")


Running hard voting ensemble on test set...
 Hard Voting Test F1: 0.8382
 Results saved to ensemble_results.csv
